In [1]:
# FEATURE ENGINEERING

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# System & paths
import sys
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Project root (one level above notebooks/)
PROJECT_ROOT = Path.cwd().parents[0]
sys.path.append(str(PROJECT_ROOT))

In [3]:
from src.eda_utils import (
    get_national_monthly_allocation,
    plot_national_allocation,
    get_zoomed_period,
    plot_allocation_anomaly
)

In [4]:
DATA_PATH = PROJECT_ROOT / "data" / "cleaned" / "primary_data_clean.csv"

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"]
)

print(df.shape)
df.head()

(67522, 16)


,country_code,state_name,state_code,dfso,district_code,full_district_code,current_year,financial_year,month,month_num,quarter,date,commodity,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
0,IN,tamil nadu,TN,Pudukkottai,PU,IN.TN.PU,2021,2021,october,10,2021Q4,2021-10-01,rice,6445.00,5595.02,NaN
1,IN,tamil nadu,TN,Pudukkottai,PU,IN.TN.PU,2021,2021,october,10,2021Q4,2021-10-01,wheat,599.00,395.49,NaN
2,IN,andhra pradesh,AP,Anantapur,AN,IN.AP.AN,2021,2021,september,9,2021Q3,2021-09-01,rice,13030.41,12247.06,NaN
3,IN,andhra pradesh,AP,Chittoor,CH,IN.AP.CH,2021,2021,september,9,2021Q3,2021-09-01,rice,14176.89,13199.87,NaN
4,IN,andhra pradesh,AP,East Godavari,EG,IN.AP.EG,2021,2021,september,9,2021Q3,2021-09-01,rice,16541.31,15778.81,NaN


In [5]:
# Ensure date is datetime 
df['date'] = pd.to_datetime(df['date'])

earliest_date = df['date'].min()
latest_date = df['date'].max()

In [6]:
print(f"Earliest date in original cleaned dataset: {earliest_date.date()}")
print(f"Latest date in original cleaned dataset: {latest_date.date()}")


Earliest date in original cleaned dataset: 2017-01-01
Latest date in original cleaned dataset: 2021-10-01


In [7]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67522 entries, 0 to 67521
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   country_code            67522 non-null  object        
 1   state_name              67522 non-null  object        
 2   state_code              67522 non-null  object        
 3   dfso                    67522 non-null  object        
 4   district_code           44053 non-null  object        
 5   full_district_code      44272 non-null  object        
 6   current_year            67522 non-null  int64         
 7   financial_year          67522 non-null  int64         
 8   month                   67522 non-null  object        
 9   month_num               67522 non-null  int64         
 10  quarter                 67522 non-null  object        
 11  date                    67522 non-null  datetime64[ns]
 12  commodity               67522 non-null  object

In [8]:
df.isna().sum()

country_code                  0
state_name                    0
state_code                    0
dfso                          0
district_code             23469
full_district_code        23250
current_year                  0
financial_year                0
month                         0
month_num                     0
quarter                       0
date                          0
commodity                     0
total_allocated_qty         396
epos_allocated_qty        22377
not_epos_allocated_qty    44753
dtype: int64

In [9]:
# STATE LEVEL FORECASTING - IGNORE DISTRICT COLUMNS
DISTRICT_COLS = [
    "dfso",
    "district_code",
    "full_district_code"
]

df = df.drop(columns=[c for c in DISTRICT_COLS if c in df.columns])

print(df.shape)

(67522, 13)


In [10]:
df.isna().sum()

country_code                  0
state_name                    0
state_code                    0
current_year                  0
financial_year                0
month                         0
month_num                     0
quarter                       0
date                          0
commodity                     0
total_allocated_qty         396
epos_allocated_qty        22377
not_epos_allocated_qty    44753
dtype: int64

In [11]:
# COLUMNS THAT GET AGGREGATED
ALLOC_COLS = [
    "total_allocated_qty",
    "epos_allocated_qty",
    "not_epos_allocated_qty"
]

In [12]:
df_state = (
    df
    .groupby(
        ["state_name", "state_code", "date", "commodity"],
        as_index=False
    )
    .agg(
        total_allocated_qty=("total_allocated_qty", "sum"),
        epos_allocated_qty=("epos_allocated_qty", "sum"),
        not_epos_allocated_qty=("not_epos_allocated_qty", "sum")
    )
)

print(df_state.shape)
df_state.head()

(2392, 7)


,state_name,state_code,date,commodity,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
0,andaman and nicobar islands,AN,2017-09-01,rice,0.10,0.02,0.0
1,andaman and nicobar islands,AN,2017-10-01,wheat,0.00,3.37,0.0
2,andaman and nicobar islands,AN,2017-11-01,rice,100.21,1040.43,0.0
3,andaman and nicobar islands,AN,2017-11-01,wheat,22.58,8.89,0.0
4,andaman and nicobar islands,AN,2017-12-01,rice,101.67,1129.41,0.0


In [13]:
# Ensure date is datetime 
df_state['date'] = pd.to_datetime(df_state['date'])

earliest_date_state = df_state['date'].min()
latest_date_state = df_state['date'].max()

In [14]:
print(f"Earliest date in state-level aggregated dataset: {earliest_date_state.date()}")
print(f"Latest date in state-level aggregated dataset: {latest_date_state.date()}")

Earliest date in state-level aggregated dataset: 2017-01-01
Latest date in state-level aggregated dataset: 2021-10-01


In [15]:
# CHECKING SEPTEMBER 2020 ISSUE
df_state[
    df_state["date"] == "2020-09-01"
].sort_values(["state_name", "commodity"]).head(10)


,state_name,state_code,date,commodity,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
63,andaman and nicobar islands,AN,2020-09-01,rice,110.05,3.13,24.44
110,andhra pradesh,AP,2020-09-01,rice,155321.51,144073.81,9.70
158,arunachal pradesh,AR,2020-09-01,rice,7466.63,1154.87,2777.96
238,assam,AS,2020-09-01,rice,115813.94,0.00,114013.94
239,assam,AS,2020-09-01,wheat,4179.00,0.00,3971.00
332,bihar,BR,2020-09-01,rice,294231.69,256718.95,0.00
333,bihar,BR,2020-09-01,wheat,196154.40,171145.58,0.00
393,chhattisgarh,CG,2020-09-01,rice,115995.99,112198.18,1516.52
480,dadra and nagar haveli and daman and diu,DH,2020-09-01,rice,1154.86,1116.96,0.00
481,dadra and nagar haveli and daman and diu,DH,2020-09-01,wheat,119.52,115.43,0.00


In [16]:
df_state["is_anomaly"] = (
    # Early incomplete / ramp-up period
    (df_state["date"] < "2018-01-01")
    
    |
    
    # COVID-era reporting shock
    (
        (df_state["date"] >= "2020-08-01") &
        (df_state["date"] <= "2020-10-01")
    )
    
    |
    
    # Incomplete tail period
    (df_state["date"] >= "2021-08-01")
).astype(int)


In [17]:
df_state.groupby("is_anomaly")["date"].agg(["min", "max", "count"])


,min,max,count
is_anomaly,,,
0,2018-01-01,2021-07-01,2013
1,2017-01-01,2021-10-01,379


In [18]:
# ADD CALENDAR FEATURES USING DATE
df_state["month_num"] = df_state["date"].dt.month
df_state["quarter"] = df_state["date"].dt.to_period("Q").astype(str)

# Financial year (Apr–Mar)
df_state["financial_year"] = np.where(
    df_state["month_num"] >= 4,
    df_state["date"].dt.year,
    df_state["date"].dt.year - 1
)


In [19]:
df_state.columns

Index(['state_name', 'state_code', 'date', 'commodity', 'total_allocated_qty', 'epos_allocated_qty',
       'not_epos_allocated_qty', 'is_anomaly', 'month_num', 'quarter', 'financial_year'],
      dtype='object')

In [20]:
df_state = df_state.rename(columns={
     "state_name": "state",
 })

In [21]:
#SORTING

# Always sort before time-based features
df_state = (
    df_state
    .sort_values(["state", "commodity", "date"])
    .reset_index(drop=True)
)


In [22]:
df_state.head(5)

,state,state_code,date,commodity,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty,is_anomaly,month_num,quarter,financial_year
0,andaman and nicobar islands,AN,2017-09-01,rice,0.10,0.02,0.0,1,9,2017Q3,2017
1,andaman and nicobar islands,AN,2017-11-01,rice,100.21,1040.43,0.0,1,11,2017Q4,2017
2,andaman and nicobar islands,AN,2017-12-01,rice,101.67,1129.41,0.0,1,12,2017Q4,2017
3,andaman and nicobar islands,AN,2018-01-01,rice,101.71,1220.90,0.0,0,1,2018Q1,2017
4,andaman and nicobar islands,AN,2018-02-01,rice,116.73,13.70,0.0,0,2,2018Q1,2017


In [23]:
FORECAST_HORIZON = 3  # 3 months
LAGS = [1, 2, 3]
ROLLING_WINDOWS = [3]

In [24]:
# TARGET CREATION
df_state = df_state.sort_values(
    ["state", "commodity", "date"]
)

df_state["target"] = (
    df_state
    .groupby(["state", "commodity"])["total_allocated_qty"]
    .shift(-FORECAST_HORIZON)
)

In [25]:
# LAG FEATURES CREATION
for lag in LAGS:
    df_state[f"lag_{lag}"] = (
        df_state
        .groupby(["state", "commodity"])["total_allocated_qty"]
        .shift(lag)
    )


In [26]:
# checking
df_state.loc[
    (df_state["state"] == "karnataka") &
    (df_state["commodity"] == "rice"),
    ["date", "total_allocated_qty", "lag_1", "lag_2", "lag_2"]
].head(10)


,date,total_allocated_qty,lag_1,lag_2,lag_2
1079,2017-08-01,253.63,NaN,NaN,NaN
1080,2017-09-01,253.84,253.63,NaN,NaN
1081,2017-10-01,23815.96,253.84,253.63,253.63
1082,2017-11-01,259727.11,23815.96,253.84,253.84
1083,2017-12-01,237024.75,259727.11,23815.96,23815.96
1084,2018-01-01,247250.54,237024.75,259727.11,259727.11
1085,2018-02-01,662160.64,247250.54,237024.75,237024.75
1086,2018-03-01,458474.70,662160.64,247250.54,247250.54
1087,2018-04-01,460255.51,458474.70,662160.64,662160.64
1088,2018-05-01,454239.68,460255.51,458474.70,458474.70


In [27]:
# ROLLING FEATURES CREATION
# roll on shifted values to avoid leakage

for window in ROLLING_WINDOWS:
    df_state[f"rolling_mean_{window}"] = (
        df_state
        .groupby(["state", "commodity"])["total_allocated_qty"]
        .shift(1)
        .rolling(window)
        .mean()
    )

    df_state[f"rolling_std_{window}"] = (
        df_state
        .groupby(["state", "commodity"])["total_allocated_qty"]
        .shift(1)
        .rolling(window)
        .std()
    )



In [28]:
# checking
df_state.loc[
    (df_state["state"] == "tamil nadu") &
    (df_state["commodity"] == "wheat"),
    ["date", "total_allocated_qty", "rolling_mean_3", "rolling_std_3"]
].head(10)


,date,total_allocated_qty,rolling_mean_3,rolling_std_3
1968,2017-09-01,27.92,NaN,NaN
1969,2017-10-01,31212.22,NaN,NaN
1970,2017-11-01,31438.60,NaN,NaN
1971,2017-12-01,30889.00,20892.913333,18069.968790
1972,2018-01-01,11920.51,31179.940000,276.218280
1973,2018-02-01,13803.13,24749.370000,11113.516622
1974,2018-03-01,6466.94,18870.880000,10450.477059
1975,2018-04-01,6929.50,10730.193333,3810.191761
1976,2018-05-01,6900.76,9066.523333,4108.536537
1977,2018-06-01,6900.96,6765.733333,259.161317


In [29]:
df_state["month"] = df_state["date"].dt.month
df_state["year"] = df_state["date"].dt.year
df_state["quarter"] = df_state["date"].dt.quarter

In [30]:
df_state.drop(['month_num','financial_year','state_code'],axis=1,inplace=True)

In [31]:
TARGET_COL = "target"

FEATURE_COLS = [
    # Lags
    "lag_1", "lag_2", "lag_3",

    # Rolling
    "rolling_mean_3",
    "rolling_std_3",

    # Calendar
    "month",
    "quarter",

    # Categorical (to encode later)
    "state",
    "commodity",
]


In [32]:
# df_model = df_state.dropna(
#     subset=FEATURE_COLS + [TARGET_COL]
# ).copy()

df_model = (
    df_state
    .dropna(subset=FEATURE_COLS + [TARGET_COL])
    .reset_index(drop=True)
)



In [33]:
df_model.columns

Index(['state', 'date', 'commodity', 'total_allocated_qty', 'epos_allocated_qty', 'not_epos_allocated_qty',
       'is_anomaly', 'quarter', 'target', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_3', 'rolling_std_3', 'month',
       'year'],
      dtype='object')

In [34]:
df_model.shape

(2052, 16)

In [35]:
df_model.head(2)

,state,date,commodity,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty,is_anomaly,quarter,target,lag_1,lag_2,lag_3,rolling_mean_3,rolling_std_3,month,year
0,andaman and nicobar islands,2018-01-01,rice,101.71,1220.9,0.0,0,1,407.87,101.67,100.21,0.10,67.326667,58.224578,1,2018
1,andaman and nicobar islands,2018-02-01,rice,116.73,13.7,0.0,0,1,414.05,101.71,101.67,100.21,101.196667,0.854712,2,2018


In [36]:
# SAVING PROCESSED DATASET
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_PATH / "state_level_features_3month_forecast.csv"

df_model.to_csv(output_path, index=False) 


In [37]:
# TRAIN_END_DATE = "2019-12-01"

# train_df = df_model[df_model["date"] <= TRAIN_END_DATE]
# val_df   = df_model[df_model["date"] > TRAIN_END_DATE]

# X_train = train_df[FEATURES]
# y_train = train_df[TARGET]

# X_val = val_df[FEATURES]
# y_val = val_df[TARGET]


In [38]:
# print("Train:", train_df["date"].min(), "→", train_df["date"].max(), X_train.shape)
# print("Val:  ", val_df["date"].min(), "→", val_df["date"].max(), X_val.shape)
